# Feature importance (I01) + batch 3 figures (validation only)

**Runtime:** Runtime → Change runtime type → **L4 GPU**. Then Runtime → Run all; about **35–45 minutes**. Keep the tab open.
If Colab disconnects, Run all again: finished parts are skipped.

1. **Batch 3 figures** (no training): walk-forward check and official-window results, drawn from the batch 3 files on Drive.
2. **Main model (E21) importance:** E21 retrained exactly as in batch 3 (scores must match), then each input shuffled
   across validation stock-days and the drop in macro-F1 recorded (3 shuffles, per-horizon best epoch).
3. **XGBoost importance:** same shuffling on the 29 price inputs, plus XGBoost's own gain importance.
4. **Importance figures and table.**

Design and expectations (fixed before any result): `claude/importance-design-2026-10-16.md`.
Outputs in `MyDrive/Senior/results/importance/` and `MyDrive/Senior/results/figures/`. Test block stays sealed.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, os, json
for f in ['src/importance.py', 'tests/test_importance.py', 'scripts/plot_results.py']:
    assert os.path.exists(f), f'stop: push {f} from your PC'
assert '--importance' in open('scripts/baselines_deep.py', encoding='utf-8').read(), 'stop: push the latest scripts/baselines_deep.py'
assert '--importance' in open('scripts/baselines_tabular.py', encoding='utf-8').read(), 'stop: push the latest scripts/baselines_tabular.py'
for f in ['scripts/baselines_deep.py', 'scripts/baselines_tabular.py', 'src/importance.py', 'scripts/plot_results.py']:
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
B3, IMP, FIG = f'{OUT_DIR}/batch3', f'{OUT_DIR}/importance', f'{OUT_DIR}/figures'
for f in [f'{B3}/e22_vs_e20_e05.json', f'{B3}/batch3_stacking.json', f'{B3}/batch3_summary.json', f'{B3}/e21_stacked.json']:
    assert os.path.exists(f), f'stop: missing {f}'
os.makedirs(IMP, exist_ok=True); os.makedirs(FIG, exist_ok=True)

In [ ]:
# 1. Batch 3 figures (no training needed)
from IPython.display import Image, display
!python scripts/plot_results.py walkforward --vs "$B3/e22_vs_e20_e05.json" --out "$FIG/batch3_walkforward.png"
!python scripts/plot_results.py official --summary "$B3/batch3_stacking.json" "$B3/batch3_summary.json" --highlight E21 --out "$FIG/batch3_official.png"
display(Image(f'{FIG}/batch3_walkforward.png')); display(Image(f'{FIG}/batch3_official.png'))

In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# Tests; the notebook stops here if any fails (about 5 minutes)
import subprocess
for t in ['tests/test_pipeline.py', 'tests/test_no_lookahead.py', 'tests/test_stopping.py', 'tests/test_importance.py',
          'tests/test_multiscale_patchtst.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-1200:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed'

In [ ]:
# 2. Main model (E21) retrained as in batch 3, with permutation importance (about 30-35 minutes)
import sys
def done(path, key):
    try:
        r = json.load(open(path))
        return all('importance' in x for x in r['models']['multiscale_sector']['runs']) and len(r['models']['multiscale_sector']['runs']) == 3 \
            if key == 'deep' else all(h in r.get('importance', {}) for h in ('h1', 'h5', 'h10'))
    except Exception:
        return False
DEEP = f'{IMP}/importance_e21.json'
if not done(DEEP, 'deep'):
    subprocess.run(['python', 'scripts/baselines_deep.py', '--models', 'multiscale_sector', '--news', 'data/derived/sentiment_scores.csv',
                    '--extra-features', 'e03', '--lr', '1e-4', '--dropout', '0.2', '--max-epochs', '30', '--patience', '10',
                    '--no-revin', '--cls-only', '--stop-rule', 'every', '--importance', '3', '--out', DEEP], check=True)
else:
    print('E21 importance: already done')
# Reproducibility: the scores must equal batch 3's E21 run (the importance option does not touch training)
a = {r['seed']: r for r in json.load(open(f'{B3}/e21_stacked.json'))['models']['multiscale_sector']['runs']}
b = json.load(open(DEEP))['models']['multiscale_sector']['runs']
diff = max(abs(r['val_per_horizon_best'][h]['macro_f1'] - a[r['seed']]['val_per_horizon_best'][h]['macro_f1'])
           for r in b for h in ('h1', 'h5', 'h10'))
print('max |difference| vs batch 3 E21:', diff, '-> REPRODUCED' if diff == 0 else '-> NOTE: differs from batch 3, tell Claude')

In [ ]:
# 3. XGBoost importance (CPU, a few minutes)
XGB = f'{IMP}/importance_xgb.json'
if not done(XGB, 'xgb'):
    subprocess.run(['python', 'scripts/baselines_tabular.py', '--extra-features', 'e03', '--importance', '3', '--out', XGB], check=True)
else:
    print('XGBoost importance: already done')

In [ ]:
# 4. Importance figures and table
!python scripts/plot_results.py importance --deep "$IMP/importance_e21.json" --xgb "$IMP/importance_xgb.json" --out-prefix "$FIG/importance" --csv "$IMP/importance_table.csv"
for f in ['importance_deep', 'importance_xgb', 'importance_groups']:
    display(Image(f'{FIG}/{f}.png'))
import pandas as pd
t = pd.read_csv(f'{IMP}/importance_table.csv')
for model in ('deep', 'xgb'):
    s = t[(t.model == model) & (t.kind == 'single')].pivot(index='input', columns='horizon', values='mean_drop')[['h1', 'h5', 'h10']]
    print(f'\n{model}: mean drop in macro-F1 when shuffled (top 10 by 10-day)')
    print(s.sort_values('h10', ascending=False).head(10).round(4).to_string())